# **Preprocesamiento**

## **División Train (80%) / Validation (20%) / Test(20%)**

In [44]:
X_raw = df[all_feature_cols]
y_clf = df["abandona"].values
y_reg = df["promedio_final"].values

# ── Un solo split stratificado por y_clf ─────────────────────────────────────
idx = np.arange(len(X_raw))

# 1) Apartar Test (20 %)
idx_temp, idx_test = train_test_split(
    idx, test_size=0.20, random_state=SEED, stratify=y_clf)

X_test_raw  = X_raw.iloc[idx_test]
y_clf_test  = y_clf[idx_test]
y_reg_test  = y_reg[idx_test]

# 2) Del 80 % restante, apartar Validation (20 % de 80 % = 16 % total)
idx_train, idx_val = train_test_split(
    idx_temp, test_size=0.20, random_state=SEED, stratify=y_clf[idx_temp])

X_train_raw = X_raw.iloc[idx_train]
X_val_raw   = X_raw.iloc[idx_val]
y_clf_train = y_clf[idx_train];  y_clf_val = y_clf[idx_val]
y_reg_train = y_reg[idx_train];  y_reg_val = y_reg[idx_val]

total = len(df)
print("División completada:")
print(f"  Train      : {len(X_train_raw):>4}  ({len(X_train_raw)/total*100:.0f} %)")
print(f"  Validation : {len(X_val_raw):>4}  ({len(X_val_raw)/total*100:.0f} %)")
print(f"  Test       : {len(X_test_raw):>4}  ({len(X_test_raw)/total*100:.0f} %)")
print("\nBalance (abandona):")
for name, y in [("Train", y_clf_train), ("Val", y_clf_val), ("Test", y_clf_test)]:
    d = y.sum(); e = (y == 0).sum()
    print(f"  {name:<10}: Dropout {d:>3} ({d/len(y)*100:.1f} %)  "
          f"Enrolled {e:>3} ({e/len(y)*100:.1f} %)")

División completada:
  Train      : 1417  (64 %)
  Validation :  355  (16 %)
  Test       :  443  (20 %)

Balance (abandona):
  Train     : Dropout 909 (64.1 %)  Enrolled 508 (35.9 %)
  Val       : Dropout 228 (64.2 %)  Enrolled 127 (35.8 %)
  Test      : Dropout 284 (64.1 %)  Enrolled 159 (35.9 %)


## **Definición del Pipeline**

In [45]:
preprocessor = ColumnTransformer(
    transformers=[
        ("ohe",    OneHotEncoder(drop="first", sparse_output=False,
                                 handle_unknown="ignore"), cat_nominal),
        ("scaler", StandardScaler(),                         numeric_vars),
        ("bool",   "passthrough",                            bool_vars),
    ],
    remainder="drop"
)

mutual_info_seeded = partial(mutual_info_classif, random_state=SEED)

def make_clf_pipeline(k, top_n):
    """
    Hiperparámetros del grid:
        k     : vecinos KNN             — {3, 5, 7, 9, 11}
        top_n : features a seleccionar — {5, 8, 10, 15}
    """
    return Pipeline([
        ("prep",   clone(preprocessor)),
        ("select", SelectKBest(mutual_info_seeded, k=top_n)),
        ("knn",    KNeighborsClassifier(n_neighbors=k, metric="euclidean")),
    ])